In [8]:
# graph_builder.py
"""
GraphBuilder — transforms a raw text document into a multi‑level, multi‑edge sparse graph
suitable for **PyTorch Geometric**.

Hierarchy
---------
word → sentence → paragraph (optional) → document

Edge families
-------------
* **seq**        : word → next‑word (within one sentence)
* **contains**   : parent → child across hierarchy levels
* **dep**        : dependency edges (optional, requires spaCy)

Design goals
------------
* Noise filter drops ~40 % tokens (one‑letter/punct/stop‑word).
* Pure Python + PyG; no external dependencies except NLTK (+spaCy if dep edges).
* Modular: choose `embed_backend = {"static", "bert"}` at build‑time.
* VRAM‑friendly: everything stays sparse; hidden dim projection = 128.
"""

from __future__ import annotations

import hashlib
import re
from dataclasses import dataclass
from typing import Dict, List, Optional

import torch
from torch import Tensor
from torch_geometric.data import HeteroData

# ─────────────────────────────────────────────────────────────────────────────
# Tokenisation (NLTK) — you can swap this for spaCy or Stanza if you prefer
# ─────────────────────────────────────────────────────────────────────────────
try:
    import nltk

    nltk.download("punkt", quiet=True)
    _sent_tokenize = nltk.sent_tokenize
    _word_tokenize = nltk.word_tokenize
except ImportError as e:
    raise ImportError("`nltk` is required: `pip install nltk`.") from e

# spaCy is **optional** — only needed when `add_syntax=True`
try:
    import spacy  # noqa: F401
except ImportError:
    spacy = None

# ─────────────────────────────────────────────────────────────────────────────
# Very small stop‑list; extend as needed.
# ─────────────────────────────────────────────────────────────────────────────
_STOP = {
    "the",
    "a",
    "an",
    "of",
    "and",
    "or",
    "to",
    "in",
    "on",
    "for",
    "with",
    "at",
    "by",
    "from",
    ",",
    ".",
    ";",
    ":",
    "'",
    "\"",
}
_RE_WS = re.compile(r"\s+")


# ─────────────────────────────────────────────────────────────────────────────
# Embedding back‑ends
# ─────────────────────────────────────────────────────────────────────────────
class _StaticTable:
    """Random static vectors (300‑d placeholder).  Swap with GloVe/fastText."""

    def __init__(self, dim: int = 300):
        self.dim = dim
        self._tbl: Dict[str, Tensor] = {}
        self._rng = torch.Generator().manual_seed(0)

    def __getitem__(self, tok: str) -> Tensor:
        if tok not in self._tbl:
            self._tbl[tok] = torch.randn(self.dim, generator=self._rng)
        return self._tbl[tok]


_STATIC_EMB = _StaticTable(dim=300)


# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _clean(s: str) -> str:
    return _RE_WS.sub(" ", s.strip())


def _keep(tok: str) -> bool:
    if tok == "I":
        return True
    if len(tok) <= 1 or not tok.isalpha():
        return False
    return tok.lower() not in _STOP


# ─────────────────────────────────────────────────────────────────────────────
# Main builder
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class GraphBuilder:
    keep_paragraphs: bool = True
    add_syntax: bool = False
    embed_backend: str = "static"  # "static" | "bert" (ctx‑emb caching stub)
    proj_dim: int = 128
    device: torch.device = torch.device("cpu")

    # simple cache so that repeated calls in same Python session are cheap
    _cache: Dict[str, Tensor] = None  # type: ignore

    def __post_init__(self):
        if self.embed_backend not in {"static", "bert"}:
            raise ValueError("embed_backend must be 'static' or 'bert'")
        if self.embed_backend == "bert":
            from transformers import AutoModel, AutoTokenizer  # lazy import

            self.bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
            self.bert = AutoModel.from_pretrained("bert-base-uncased").eval().to("cpu")
        self._cache = {}
        self._proj = torch.nn.Linear(300 if self.embed_backend == "static" else 768, self.proj_dim, bias=False)

    # ─────────────────────────────────────────────────────────────────────
    # Public entry
    # ─────────────────────────────────────────────────────────────────────
    def build(self, text: str) -> HeteroData:
        text = _clean(text)
        data = HeteroData()

        # 1 — Document node (single)
        data["doc"].x = torch.zeros(1, self.proj_dim, device=self.device)
        doc_id = 0

        sent_global_ids: List[int] = []
        para_global_ids: List[int] = []

        # 2 — Parse paragraphs & sentences
        paragraphs = text.split("\n\n") if self.keep_paragraphs else [text]
        for para in paragraphs:
            sent_ids_this_para: List[int] = []
            for sent in _sent_tokenize(para):
                tokens = [t for t in _word_tokenize(sent) if _keep(t)]
                if not tokens:
                    continue

                # word nodes
                w_offset = data["word"].num_nodes if "word" in data else 0
                word_node_ids = list(range(w_offset, w_offset + len(tokens)))
                word_feats = torch.stack([self._get_word_vec(t) for t in tokens])  # shape (n, d_in)
                data["word"].x = (
                    torch.cat([data["word"].x, self._proj(word_feats).to(self.device)])
                    if "word" in data
                    else self._proj(word_feats).to(self.device)
                )

                # sequential edges word→word
                if len(word_node_ids) > 1:
                    _add_edges(data, "word", "word", "seq", word_node_ids[:-1], word_node_ids[1:])

                # sentence node
                s_id = data["sent"].num_nodes if "sent" in data else 0
                sent_vec = word_feats.mean(dim=0, keepdim=True)  # (1, d_in)
                data["sent"].x = (
                    torch.cat([data["sent"].x, self._proj(sent_vec).to(self.device)])
                    if "sent" in data
                    else self._proj(sent_vec).to(self.device)
                )
                sent_ids_this_para.append(s_id)
                sent_global_ids.append(s_id)

                # containment sent→word
                _add_edges(data, "sent", "word", "contains", [s_id] * len(word_node_ids), word_node_ids)

            # paragraph node (optional)
            if self.keep_paragraphs and sent_ids_this_para:
                p_id = data["para"].num_nodes if "para" in data else 0
                para_vec = data["sent"].x[sent_ids_this_para].mean(dim=0, keepdim=True)
                data["para"].x = (
                    torch.cat([data["para"].x, para_vec]) if "para" in data else para_vec
                )
                para_global_ids.append(p_id)
                _add_edges(data, "para", "sent", "contains", [p_id] * len(sent_ids_this_para), sent_ids_this_para)

        # 3 — Doc→(para|sent) containment
        if self.keep_paragraphs and para_global_ids:
            _add_edges(data, "doc", "para", "contains", [doc_id] * len(para_global_ids), para_global_ids)
        else:
            _add_edges(data, "doc", "sent", "contains", [doc_id] * len(sent_global_ids), sent_global_ids)

        # 4 — Dependency edges (optional, expensive)
        if self.add_syntax:
            if spacy is None:
                raise ImportError("spaCy required for syntactic edges; `pip install spacy`.")
            nlp = spacy.load("en_core_web_sm")
            parsed = nlp(text)
            wp_lookup: List[int] = list(range(data["word"].num_nodes))  # quick map
            dep_src, dep_dst = [], []
            for tok in parsed:
                if not _keep(tok.text):
                    continue
                head = tok.head
                if head.i == tok.i or not _keep(head.text):
                    continue
                dep_src.append(wp_lookup[head.i])
                dep_dst.append(wp_lookup[tok.i])
            if dep_src:
                _add_edges(data, "word", "word", "dep", dep_src, dep_dst)

        return data

    # ─────────────────────────────────────────────────────────────────────
    # Embedding helpers
    # ─────────────────────────────────────────────────────────────────────
    def _get_word_vec(self, tok: str) -> Tensor:
        if self.embed_backend == "static":
            return _STATIC_EMB[tok]
        # —— BERT path (contextual) ——
        h = hashlib.sha1(tok.lower().encode()).hexdigest()
        if h in self._cache:
            return self._cache[h]
        # fall back to static if we don't have context (word‑level BERT too slow)
        vec = _STATIC_EMB[tok]
        self._cache[h] = vec
        return vec


# ─────────────────────────────────────────────────────────────────────────────
# Edge utility (safe w.r.t. PyG internals)
# ─────────────────────────────────────────────────────────────────────────────

def _add_edges(data: HeteroData, src_t: str, dst_t: str, rel: str, src: List[int], dst: List[int]):
    """Append edges; create edge store if absent."""
    key = (src_t, rel, dst_t)
    ei = torch.tensor([src, dst], dtype=torch.long)
    store = data[key]  # ⟵ auto‑creates empty edge store
    if "edge_index" in store:
        store.edge_index = torch.cat([store.edge_index, ei], dim=1)
    else:
        store.edge_index = ei


# ─────────────────────────────────────────────────────────────────────────────
# Quick manual test
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    sample = (
        "I prefer the morning flight through Denver. "
        "Connections are short, and the service is excellent.\n\n"
        "However, delays are common in the afternoon flights."
    )
    gb = GraphBuilder()
    g = gb.build(sample)
    print(g)
    for k, v in g.edge_index_dict.items():
        print(f"{k}: {v.shape[1]} edges")


HeteroData(
  doc={ x=[1, 128] },
  word={ x=[6, 128] },
  sent={ x=[1, 128] },
  para={ x=[1, 128] },
  (word, seq, word)={ edge_index=[2, 15] },
  (sent, contains, word)={ edge_index=[2, 18] },
  (para, contains, sent)={ edge_index=[2, 3] },
  (doc, contains, para)={ edge_index=[2, 1] }
)
('word', 'seq', 'word'): 15 edges
('sent', 'contains', 'word'): 18 edges
('para', 'contains', 'sent'): 3 edges
('doc', 'contains', 'para'): 1 edges


In [9]:
# viz_graph.py  ── minimal Graphviz renderer for a PyG HeteroData graph
#
# Requires:  pip install graphviz           # Python bindings
#            sudo apt-get install graphviz  # or brew install graphviz
#
# Usage:
#     from graph_builder import GraphBuilder
#     from viz_graph import hetero_to_graphviz
#
#     gb = GraphBuilder()
#     gdata = gb.build(text)
#     dot = hetero_to_graphviz(gdata, max_nodes=120)   # returns graphviz.Digraph
#     dot.render("example_graph", format="pdf", view=True)

from graphviz import Digraph
from torch_geometric.data import HeteroData


def hetero_to_graphviz(
    data: HeteroData,
    max_nodes: int | None = 200,
    plain: bool = False,
) -> Digraph:
    """
    Convert a heterogeneous PyG graph to a Graphviz Digraph.

    Parameters
    ----------
    data : HeteroData
        Output from GraphBuilder (or any PyG hetero graph).
    max_nodes : int | None
        Hard cap to avoid huge diagrams; `None` = draw all nodes.
    plain : bool
        If True, render every node as a gray circle (no type colour-coding).

    Returns
    -------
    graphviz.Digraph
    """
    g = Digraph("G")
    g.attr(rankdir="LR", fontsize="10", margin="0.05")

    # --- assign a colour per node type (light palettes for readability)
    _palette = [
        "#AED6F1",  # doc
        "#A9DFBF",  # para
        "#FAD7A0",  # sent
        "#F5B7B1",  # word
        "#D7BDE2",  # other
    ]
    colour_map = {}
    for idx, ntype in enumerate(sorted(data.node_types)):
        colour_map[ntype] = _palette[idx % len(_palette)]

    # --- add nodes
    node_counter = 0
    for ntype in data.node_types:
        num = data[ntype].num_nodes
        for i in range(num):
            if max_nodes is not None and node_counter >= max_nodes:
                break
            nid = f"{ntype}_{i}"
            label = f"{ntype}\\n#{i}"
            g.node(
                nid,
                label=label,
                shape="oval" if ntype == "word" else "box",
                style="filled",
                fillcolor=colour_map[ntype] if not plain else "#DDDDDD",
            )
            node_counter += 1

    # --- add edges
    for (src_type, rel, dst_type), store in data.edge_index_dict.items():
        src, dst = store
        n_edges = src.numel()
        for k in range(n_edges):
            if max_nodes is not None and (src[k] >= max_nodes or dst[k] >= max_nodes):
                continue
            g.edge(
                f"{src_type}_{src[k].item()}",
                f"{dst_type}_{dst[k].item()}",
                label=rel,
                fontsize="8",
            )

    return g


# quick smoke-test -----------------------------------------------------------
if __name__ == "__main__":


    sample = (
        "I prefer the morning flight through Denver. "
        "Connections are short, and the service is excellent.\n\n"
        "However, delays are common in the afternoon flights."
    )

    gb = GraphBuilder()
    graph = gb.build(sample)

    dot = hetero_to_graphviz(graph, max_nodes=80)
    # Write PDF and open it
    dot.render("sample_doc_graph", format="pdf", view=True)
